In [0]:
#POC for health adjudication

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp

base_path = "/Volumes/healthcare_adjudication/healthcare_adjudication_sch/healthcare_adjudication_vol"

bronze_schema = "healthcare_adjudication.healthcare_adjudication_bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

In [0]:
providers_schema = StructType([
    StructField("provider_id", IntegerType()),
    StructField("provider_name", StringType()),
    StructField("specialty", StringType()),
    StructField("state", StringType())
])

providers = (
    spark.read.format("csv")
    .option("header","true")
    .schema(providers_schema)
    .load(f"{"/Volumes/healthcare_adjudication/healthcare_adjudication_sch/healthcare_adjudication_vol"}/providers.csv")
    .withColumn("ingestion_time", current_timestamp())
)
providers.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.providers")



payers table

In [0]:
payers_schema = StructType([
    StructField("payer_id", IntegerType()),
    StructField("payer_name", StringType()),
    StructField("plan_type", StringType())
])

payers = (
    spark.read.format("csv")
    .option("header","true")
    .schema(payers_schema)
    .load(f"{base_path}/payers.csv")
    .withColumn("ingestion_time", current_timestamp())
)

payers.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.payers")


Patients

In [0]:
patients = (
    spark.read.option('multiLine', True).json(f"{base_path}/patients.json")
    .withColumn("ingestion_time", current_timestamp())
)

patients.write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.patients")


policies

In [0]:
policies = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/policies_nested.json")
    .withColumn("ingestion_time", current_timestamp())
)

policies.write.format("delta").mode("append") \
.saveAsTable("healthcare_adjudication.healthcare_adjudication_bronze.policies")


Claim_Lines

In [0]:
claim_lines = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/claim_lines.json")
    .withColumn("ingestion_time", current_timestamp())
)

claim_lines.write.format("delta").mode("append") \
.saveAsTable("healthcare_adjudication.healthcare_adjudication_bronze.claim_lines")


Claims

In [0]:
claims_schema = StructType([
    StructField("claim_id", IntegerType()),
    StructField("patient_id", IntegerType()),
    StructField("provider_id", IntegerType()),
    StructField("policy_id", IntegerType()),
    StructField("claim_amount", DoubleType()),
    StructField("claim_date", DateType()),
    StructField("status", StringType())
])

claims = (
    spark.read.format("csv")
    .option("header","true")
    .schema(claims_schema)
    .load(f"{base_path}/claims.csv")
    .withColumn("ingestion_time", current_timestamp())
)

claims.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.claims")


Payments

In [0]:
payments = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/payments.json")
    .withColumn("ingestion_time", current_timestamp())
)

payments.write.format("delta").mode("append") \
.saveAsTable("healthcare_adjudication.healthcare_adjudication_bronze.payments")


In [0]:
%sql
--ALTER SCHEMA healthcare_adjudication_bronze RENAME TO hc_adjudication

In [0]:
%sql
use catalog healthcare_adjudication



In [0]:
%sql
use schema healthcare_adjudication_bronze

In [0]:
%sql
select * from providers

In [0]:
%sql select * from payers

In [0]:
%sql
SELECT * FROM patients LIMIT 20

In [0]:
%sql
select * from policies limit 20

In [0]:
%sql
select * from claim_lines limit 10

In [0]:
%sql
select * from claims limit 10

In [0]:

policies.printSchema()